loading the copy of data set


In [2]:
import pandas as pd


In [5]:
df_original = pd.read_csv("data/raw.csv")
df = df_original.copy()

print("original dataset shape : ", df_original.shape)


original dataset shape :  (10000, 14)


CHECK AND HANDLE MISSING VALUES

In [6]:
print(df.isnull().sum())

UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64


In [7]:
missing_columns = df.columns[df.isnull().any()].tolist()

if missing_columns:
    print("\nColumns containing missing values:", missing_columns)
    print("Investigate these values before deciding how to handle them.")
else:
    print("\nNo missing values found.")


No missing values found.


CHECKING DUPLICATE ROWS

In [8]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


VALIDATE CATEGORICAL VALUES

In [9]:
if "Type" in df.columns:
    print("Observed product types:")
    print(df["Type"].value_counts(dropna=False))

    expected_types = {"L", "M", "H"}
    observed_types = set(df["Type"].dropna().unique())

    unexpected_types = observed_types - expected_types
    print("Unexpected product types:", unexpected_types)

Observed product types:
Type
L    6000
M    2997
H    1003
Name: count, dtype: int64
Unexpected product types: set()


VALIDATE BINARY FAILURE LABELS

In [10]:
label_columns = [
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

for column in label_columns:
    if column in df.columns:
        print(f"\n{column} distribution:")
        print(df[column].value_counts(dropna=False))

        observed_values = set(df[column].dropna().unique())
        unexpected_values = observed_values - {0, 1}

        if unexpected_values:
            print("WARNING: Unexpected label values:", unexpected_values)
        else:
            print("Binary values validated (0 and 1 only).")


Machine failure distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64
Binary values validated (0 and 1 only).

TWF distribution:
TWF
0    9954
1      46
Name: count, dtype: int64
Binary values validated (0 and 1 only).

HDF distribution:
HDF
0    9885
1     115
Name: count, dtype: int64
Binary values validated (0 and 1 only).

PWF distribution:
PWF
0    9905
1      95
Name: count, dtype: int64
Binary values validated (0 and 1 only).

OSF distribution:
OSF
0    9902
1      98
Name: count, dtype: int64
Binary values validated (0 and 1 only).

RNF distribution:
RNF
0    9981
1      19
Name: count, dtype: int64
Binary values validated (0 and 1 only).


VALIDATE NUMERIC SENSOR COLUMNS

In [11]:
sensor_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

for column in sensor_columns:
    if column in df.columns:
        print(f"\n{column}")
        print("Data type:", df[column].dtype)
        print("Minimum:", df[column].min())
        print("Maximum:", df[column].max())
        print("Negative values:", (df[column] < 0).sum())

        if not pd.api.types.is_numeric_dtype(df[column]):
            print("WARNING: Column is not numeric.")
    else:
        print(f"WARNING: Missing expected column: {column}")



Air temperature [K]
Data type: float64
Minimum: 295.3
Maximum: 304.5
Negative values: 0

Process temperature [K]
Data type: float64
Minimum: 305.7
Maximum: 313.8
Negative values: 0

Rotational speed [rpm]
Data type: int64
Minimum: 1168
Maximum: 2886
Negative values: 0

Torque [Nm]
Data type: float64
Minimum: 3.8
Maximum: 76.6
Negative values: 0

Tool wear [min]
Data type: int64
Minimum: 0
Maximum: 253
Negative values: 0


CHECKING TEMPERATURE RELATIONSHIP

In [ ]:
air_col = "Air temperature [K]"
process_col = "Process temperature [K]"

if air_col in df.columns and process_col in df.columns:
    suspicious = df[process_col] <= df[air_col]
    print("Rows where process temperature <= air temperature:",
        suspicious.sum())


Rows where process temperature <= air temperature: 0


REORDERING The COLUMNS

In [13]:
column_order = [
    "UDI",
    "Product ID",
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
]

# Keep any additional columns that might exist.
ordered_columns = [
    column for column in column_order if column in df.columns
]
remaining_columns = [
    column for column in df.columns if column not in ordered_columns
]

df = df[ordered_columns + remaining_columns]

print("Reordered columns:")
print(df.columns.tolist())

Reordered columns:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']


FINAL DATA QUALITY REPORT

In [14]:
print("Original shape:", df_original.shape)
print("Current shape:", df.shape)
print("Missing cells:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

if "UDI" in df.columns:
    print("UDI unique:", df["UDI"].is_unique)

print("\nFinal data types:")
print(df.dtypes)

Original shape: (10000, 14)
Current shape: (10000, 14)
Missing cells: 0
Duplicate rows: 0
UDI unique: True

Final data types:
UDI                          int64
Product ID                  object
Type                        object
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object


SAVING WORKING DATASET

In [15]:
df.to_csv("process_data.csv", index=False)

print("\nValidated dataset saved as process_data.csv")



Validated dataset saved as process_data.csv
